04 — Train a small classifier (ResNet18 → chihuahua / muffin)
=============================================================
Goal: fine-tune only the last layer of ImageNet-pretrained ResNet18 so it
answers two classes: chihuahua vs muffin.

Why transfer learning?
  Training a CNN from scratch needs lots of data and time.
  ResNet18 already knows generic visual features; we only retrain the head.

Expected:
  - Device: cuda (gpu-profile) or cpu
  - Classes: ['chihuahua', 'muffin']
  - After ~3 epochs: test_acc often > 0.90 on this lab subset

Output:
  /opt/app-root/src/models/model.pt  (PyTorch checkpoint)

Important: keep this notebook kernel alive — 05 needs the in-memory `model`.

In [ ]:
import time
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms

# ---------------------------------------------------------------------------
# Paths & device
# ---------------------------------------------------------------------------
DATA = Path("/opt/app-root/src/data/muffin-chihuahua")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

# ---------------------------------------------------------------------------
# Transforms
#   - resize to 224×224 (ResNet input)
#   - ImageNet normalization (must match pretrained weights)
#   - light augmentation on train only
# ---------------------------------------------------------------------------
train_tf = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]
)
test_tf = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]
)

# ImageFolder expects: DATA/train/<class>/*.jpg
train_ds = datasets.ImageFolder(DATA / "train", transform=train_tf)
test_ds = datasets.ImageFolder(DATA / "test", transform=test_tf)
print("Classes:", train_ds.classes)  # expect ['chihuahua', 'muffin']
print("Train images:", len(train_ds), "| Test images:", len(test_ds))

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=2)

# ---------------------------------------------------------------------------
# Model: freeze backbone, replace final Linear with 2-class head
# ---------------------------------------------------------------------------
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
for param in model.parameters():
    param.requires_grad = False
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
# Optimize ONLY the new head
optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3)


def evaluate() -> float:
    """Return accuracy on the test set."""
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            preds = model(images).argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return correct / total


# ---------------------------------------------------------------------------
# Train a few epochs (enough for the lab demo)
# ---------------------------------------------------------------------------
EPOCHS = 3
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    t0 = time.time()

    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    acc = evaluate()
    avg_loss = running_loss / len(train_loader)
    print(
        f"epoch {epoch + 1}/{EPOCHS}  "
        f"loss={avg_loss:.4f}  test_acc={acc:.3f}  "
        f"time={time.time() - t0:.1f}s"
    )

# ---------------------------------------------------------------------------
# Save checkpoint for later export / debugging
# ---------------------------------------------------------------------------
out_dir = Path("/opt/app-root/src/models")
out_dir.mkdir(parents=True, exist_ok=True)
ckpt = out_dir / "model.pt"
torch.save({"model": model.state_dict(), "classes": train_ds.classes}, ckpt)
print("Saved", ckpt)
print("\nOK — continue with 05_export_onnx_upload.py (keep `model` in memory)")